# Étape 1 — Exploration et Nettoyage des données (EDA)

Objectif : charger les données, identifier les anomalies et nettoyer le jeu de données.

## 1.1 Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

df = pd.read_parquet('../data/yellow_tripdata.parquet')
df = df.sample(n=100_000, random_state=42).reset_index(drop=True)
print(f'Shape : {df.shape}')
df.head()

## 1.2 Structure et types

In [ ]:
df.info()
df.describe()

## 1.3 Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

## 1.4 Détection et suppression des valeurs aberrantes

Filtres appliqués :
- `trip_distance` > 0 et < 100 miles
- `fare_amount` > 0 et < 500 $
- `passenger_count` entre 1 et 6
- Durée positive (après calcul)

In [ ]:
df_clean = df.copy()

df_clean = df_clean[
    (df_clean['trip_distance'] > 0) & (df_clean['trip_distance'] < 100) &
    (df_clean['fare_amount'] > 0) & (df_clean['fare_amount'] < 500) &
    (df_clean['passenger_count'] >= 1) & (df_clean['passenger_count'] <= 6)
]

print(f'Lignes avant : {len(df)} — après : {len(df_clean)} ({len(df)-len(df_clean)} supprimées)')

## 1.5 Visualisations exploratoires

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_clean['trip_distance'].hist(bins=50, ax=axes[0,0])
axes[0,0].set_title('Distribution des distances')

df_clean['fare_amount'].hist(bins=50, ax=axes[0,1], color='orange')
axes[0,1].set_title('Distribution des tarifs')

df_clean['payment_type'].value_counts().plot(kind='bar', ax=axes[1,0], color='green')
axes[1,0].set_title('Modes de paiement')

df_clean['passenger_count'].value_counts().sort_index().plot(kind='bar', ax=axes[1,1], color='purple')
axes[1,1].set_title('Nombre de passagers')

plt.tight_layout()
plt.savefig('../figures/01_eda_distributions.png', dpi=150)
plt.show()

In [ ]:
df_clean.to_parquet('../data/yellow_tripdata_clean.parquet', index=False)
print('Données nettoyées sauvegardées.')